In [3]:
# SETUP REQUIRED: Please run these commands in your terminal to create the conda environment:
#
# conda install mamba
# export MAMBA_DOWNLOAD_TIMEOUT_SECONDS=600
# mamba create -n openff-toolkit -c conda-forge openff-toolkit
# conda activate openff-toolkit
# pip install biopython mendeleev nbval numpy==2.4.3 pandas==2.3.3 pdbfixer pytest scipy==1.17.1 matplotlib==3.10.8 ipykernel packmol MDAnalysis
# python -m ipykernel install --user --name openff-toolkit --display-name "openff-toolkit"
#
# Then select the "openff-toolkit" kernel in VS Code
#
# See MoBi/installation.txt for details

# general
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# to fix Bolts OXT
from pdbfixer import PDBFixer

# OFF
from openff.toolkit import Molecule, Topology
from openff.units import unit as off_unit
from openff.toolkit import ForceField as OFF_ForceField
from openff.interchange import Interchange
from openff.interchange.components._packmol import pack_box

# OMM
from openmm import LangevinIntegrator, Vec3
from openmm.app import Simulation, PDBReporter, StateDataReporter, PDBFile
from openmm.app import Modeller
from openmm.app import ForceField as OMM_ForceField
from openmm import unit as omm_unit

# to make water
from scipy.constants import N_A
from mendeleev import element

# warnings
import warnings
from openff.interchange.warnings import InterchangeCombinationWarning

# configs
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=InterchangeCombinationWarning)  # trustmebro

# Inspiration
# https://pubs.acs.org/doi/10.1021/acs.jcim.1c00998
# https://docs.openforcefield.org/en/latest/examples/openforcefield/openff-interchange/protein_ligand/protein_ligand.html
# https://github.com/openforcefield/openff-interchange/blob/v0.5.2/examples/protein_ligand/protein_ligand.ipynb

ENV = os.path.dirname(os.path.dirname(sys.executable))

# link to Packmol
packmol_dir = str(Path(ENV) / "bin")
os.environ["PATH"] = os.pathsep.join([os.environ.get("PATH", ""), packmol_dir])

# intentional hardcode for reproducibility in a fragile env
#env_site_packages = ENV + "/lib/python3.12/site-packages"

ModuleNotFoundError: No module named 'pdbfixer'

# Input
Here we define the input file:
- a protein `.pdb` file. This should contain your protein, but no crystal water, no cofactors or other small molecules
- (optional) a ligand `.sdf` file, for example from a docking experiment

In [ ]:
WORKDIR = Path.cwd()

input_file = WORKDIR / "1uyd_protein.pdb"
ligand_file = ''

basename = Path(input_file).with_suffix("").name

# Protein
First we fix the `pdb` structure, and add missing atoms and residues. This is specifically needed for the `OXT` marker.

In [ ]:
fixer = PDBFixer(str(input_file))
fixer.findMissingResidues()  # locate missing residues (required for OXT)
fixer.findMissingAtoms()  # adds whole residues if any are absent
fixer.addMissingAtoms()
PDBFile.writeFile(fixer.topology, fixer.positions, open(f"{basename}_fixed.pdb", "w"))

### Protein Forcefield (OMM)
Now we want to map a ForceField to our protein. Therefore we load in the fixed protein `.pdb` file and define the topology and positions. We then add Hydrogens and save the result as a `.pdb` file.

In [ ]:
pdb = PDBFile(f"{basename}_fixed.pdb")
modeller = Modeller(pdb.topology, pdb.positions)

forcefield = OMM_ForceField(
    "amber14-all.xml",
    "tip3p.xml",
    #env_site_packages + "/openforcefields/offxml/openff-2.2.0.offxml",
)

modeller.addHydrogens(forcefield, pH=7)

PDBFile.writeFile(
    modeller.topology, modeller.positions, open(f"{basename}_protein_H.pdb", "w")
)

### Protein Forcefield (OFF)
Now we take the protein with hydrogens and translate to a ForceField that is capable of dealing with small molecules. We first extract all protein molecules (= individual chains) and then translate using a so called interchange object.

In [ ]:
protein_with_hydrogens = Topology.from_pdb(f"{basename}_protein_H.pdb")

protein_molecules = [
    protein_with_hydrogens.molecule(n)
    for n in range(protein_with_hydrogens.n_molecules)
]

protein_interchange = Interchange.from_smirnoff(
    force_field=OFF_ForceField("ff14sb_off_impropers_0.0.4.offxml"),
    topology=protein_molecules,
)

# Ligand
In case we have a ligand file, we want to load it and create a molecule interchange object:

In [ ]:
if ligand_file:
    molecules = Molecule.from_file(ligand_file)

    ligand_interchange = Interchange.from_smirnoff(
        force_field=OFF_ForceField(
            "openff_unconstrained-2.0.0.offxml"
        ),  # ligand force field
        topology=molecules,
    )

## Combine Protein and Ligand
If we have a ligand, we want to combine the protein and ligand interchange objects and update the topology.
If we do not have a ligand, we just rename the variables to use in the next steps.

In [ ]:
if ligand_file:
    docked_interchange = protein_interchange.combine(ligand_interchange)
    docked_topology = Topology.from_molecules(protein_molecules + molecules)
else:
    docked_interchange = protein_interchange
    docked_topology = Topology.from_molecules(protein_molecules)

## Make the Simulation Box
To run the MD simulation, we need a box filled with water to put the protein in. So first we need to decide on the box dimensions. For that we take the entire solute (protein + ligand) coordinates, and center it in the imaginary xyz coordinate system. We calculate the (positive and negative) maximum extension, add a padding of 2 nanometer and that gives us the box.

This is not the best method and for longer MD simulations, more considerations should go into the box and environment!

In [ ]:
coords = [
    docked_topology.molecule(i).conformers[0]
    for i in range(docked_topology.n_molecules)
]
xyz = np.vstack(coords)

centroid = (
    xyz.sum(axis=0) / xyz.shape[0]
)

centered = xyz.to(off_unit.nanometer) - centroid
mins = centered[:, :3].magnitude.min(axis=0)
maxs = centered[:, :3].magnitude.max(axis=0)

padding = 2 # nanometer
aligned_box_vectors = np.diag(maxs - mins + padding) * off_unit.nanometer

### Fill with Water
Now that we have the box dimensions, we need to figure out how much water to add. We du this using Avogadro's number (`N_A`) and the molecular mass of $H_2O$.

In [ ]:
volume = float(
    (aligned_box_vectors.diagonal().prod()).to(off_unit.centimeter**3).magnitude
)

mass_H2O = 2 * element("H").atomic_weight + element("O").atomic_weight # this assumes a rho of 1

# number of TIP3P water molecules for a density of 1 g·cm⁻³
n_water = int(round(volume * N_A / mass_H2O))

### Add Ions
Aside from $H_2O$ we also need ions to balance the charge of the solute. We calculate the total charge and the respective amount of $Na^+$  and $Cl^-$.

In [ ]:
total_charge = round(sum(docked_interchange["Electrostatics"].charges.values()), 3)

solvent_mols = [Molecule.from_smiles(s) for s in ("O", "[Cl-]", "[Na+]")]

for mol in solvent_mols:
    mol.generate_conformers(n_conformers=1)

# Amount of copies of each solvent molecule
copies = [n_water, max(0, int(total_charge.m)), max(0, -int(total_charge.m))]

## Construct the Box
Now that we have the number of waters, sodium and chloride, we can create the solvent interchange and construct the box using Packmol. We save the water box as `.pdb` file again.

In [ ]:
water_interchange = Interchange.from_smirnoff(
    force_field=OFF_ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=[solvent_mols[0]] * copies[0]
    + [solvent_mols[2]] * copies[2]
    + [solvent_mols[1]] * copies[1],
)

packed_topology = pack_box(
    molecules=solvent_mols,
    number_of_copies=copies,
    solute=docked_topology,
    box_vectors=aligned_box_vectors,
    center_solute=True,
)

packed_topology.to_file(f"{basename}_packed.pdb")

## Combine Solvent and Solute Topology
Now we combine the solute and the water box into the system interchange. Now everything is in one object:

In [ ]:
system_interchange = docked_interchange.combine(water_interchange)
# add positions back
system_interchange.positions = packed_topology.get_positions()
system_interchange.box = packed_topology.box_vectors

In [ ]:
# export to openMM
openmm_topology = system_interchange.topology.to_openmm(ensure_unique_atom_names=False)
openmm_system = system_interchange.to_openmm()

## Prepare the Simulation
Now we prepare the simulation object with all variables that we need.

In [ ]:
integrator = LangevinIntegrator(
    300 * omm_unit.kelvin,
    1 / omm_unit.picosecond, #friction coefficient
    0.002 * omm_unit.picoseconds #stepsize
)
simulation = Simulation(openmm_topology, openmm_system, integrator)

In [ ]:
# update positions
pos_array = np.asarray(system_interchange.positions / omm_unit.nanometer, dtype=float)
vecs = [
    Vec3(float(x), float(y), float(z)) for x, y, z in pos_array
] * omm_unit.nanometer

simulation.context.setPositions(vecs)

Execute the next cell to do a quick minimization:

In [ ]:
simulation.minimizeEnergy()

Execute the next cell to run 10000 steps of Molecular Dynamics:
Look at the trajectory in `_output.pdb`.

In [ ]:
log_file = f"{basename}_log.txt"
simulation.reporters.append(PDBReporter(f"{basename}_output.pdb", 100))
simulation.reporters.append(
    StateDataReporter(log_file, 100, step=True, potentialEnergy=True, temperature=True)
)

simulation.step(10000)